## Before running: database, operating system, and existing outputs

This notebook's main parameter cell can update saved timing corrections and
replace `data/on_list_times.npy`. Read this before running it.

**Reviewing the timing plots after the calculation finishes is a critical
step. Inspect the first/last stimuli and every unusual or repaired interval
against the photodiode trace before RF generation. A matching count alone
does not establish correct timing. Apply the same review to every preview.**

**Linux/macOS:** keep the current database connection in
`Utils/session_edits.py`, which uses `unix-dotfile` locking.
Start Jupyter with the README's `RF_MOUSE_DIR`, `RF_DATE`, and `RF_SESSION`
environment variables set. The parameter cell reads these values. Paths must
be visible to the machine running the kernel. A Windows browser connected to
Linux still uses Linux paths.

**Native Windows:** set `$env:RF_MOUSE_DIR = "D:\recordings\mouse_01"` in PowerShell before starting Jupyter.
In your Windows checkout, inside `SessionEditStore._connect` in
`Utils/session_edits.py`, replace:

```python
database_uri = f"{database_path.as_uri()}?vfs=unix-dotfile"
```

with:

```python
database_uri = database_path.as_uri()
```

Keep the existing `sqlite3.connect(database_uri, uri=True, timeout=30)`
and the rest of the function unchanged, then restart the kernel. This
connection change is for native Windows; keep the original for the Linux/macOS
lab setup. It is not a claim that every upstream sorting command supports
Windows.

**Preserve the existing database.** SQLite files are portable across platforms,
but this connection's locking method is OS-specific. Before opening
`<mouse directory>/session_slices.sqlite3`:

- Confirm this is the existing file for the correct mouse. A missing or mistyped
  path causes `check_session_edits()` to create a new, empty database.
- Close database users on all computers and make a uniquely named backup.
  Never overwrite another database when copying. Do not delete/recreate the
  original to fix a Windows connection or legacy-schema error.
- Do not access one shared database concurrently through Windows and
  Linux/macOS or different database-browser locking modes.
- Preserve the existing onset file before a rerun. `is_overwrite_parameters`
  does **not** protect the database or onset output.
- Run the cells individually. The main timing cell writes correction records
  and the onset file; it has no preview switch. Inspect the following first/last
  time checks, unusual intervals, and photodiode plot before using that saved
  file for RF generation. Change the plot's time range to inspect the start
  and every unusual or repaired interval, not only the default end view.

See [Platform setup and preserving existing data](docs/platforms_and_data_safety.md)
for PowerShell setup, database paths, and a backup that refuses overwrite.
Follow the [README](README.md) for raw inputs → intermediate files → RF.


In [ ]:
%matplotlib inline
# %matplotlib ipympl
%load_ext autoreload
%autoreload 2

import os
import numpy as np

from Utils.load_files import load_binary
from Utils.recording import detect_exposure_time
from pathlib import Path
import matplotlib
# Restart the kernel to revert changes to matplotlib settings
# matplotlib.use('TkAgg')  # Use TkAgg backend for interactive plotting

import matplotlib.pyplot as plt
from Utils.Sessions import Session
from Utils.ttl_utils import fill_short_gaps, gen_delete_by_range_index
from scipy.io import loadmat

from Utils.recording import interp_replace
from Utils.session_edits import check_session_edits



In [ ]:
"""
Change all the variables below to match recording.
"""

is_overwrite_parameters: bool = True
base_dir = os.environ["RF_MOUSE_DIR"]
date: str | int = os.environ["RF_DATE"]
num_of_rec: int = int(os.environ["RF_SESSION"])

subfolder_filler = f"{date}_{num_of_rec}"

sessionEdit = check_session_edits(
    f"{base_dir}/session_slices.sqlite3",
    date,
    num_of_rec,
)

base_dir = f"{base_dir}/{date}/{subfolder_filler}"

# These are 0 indexed channel numbers
analogue_input_channel: int = 3

is_convert_to_zero: bool = False
time_interval: bool = False
is_signal_inverted: bool = True

detect_ADC_DI_threshold, fill_short_gaps_1, fill_short_gaps_2 = 15000, 10, 1000
detect_on_list_threshold: float = 0.5

parameters_file = Path(f'{base_dir}/data/parameters.json')

"""------------------------------------------------------------------"""
oebin_file_dir = next((Path(base_dir) / date).glob("*/experiment1/recording1/structure.oebin"))

session = Session(oebin_file_dir)
session_info = session.get_session_info()

base_data_dir = session_info['base_path']
record_nodes: str = session_info['record_nodes']
recording_name: str = session_info['recording_name']
experiment_id: str = session_info['experiment_id']

path_between = f'/{record_nodes}/{experiment_id}/'
continuous_folder = base_data_dir + path_between + recording_name + '/continuous/'
print("==============================")
print("Loading ADC npy data...")
deleteStartIndex = None
deleteEndIndex = None
deleteFrames, interp_start, interp_end, frames_between = None, None, None, None

ADC_name: str = session_info['continuous_ADC_folder']
analogue_total_input_channel_number: int = session_info['ADC_input_channel']
ADC_sampling_rate: float = session_info['ADC_sample_rate']
ADC_datafile = continuous_folder + ADC_name + '/continuous.dat'

photodiode_data = load_binary(ADC_datafile, analogue_total_input_channel_number, analogue_input_channel)

ADC_continuous_time_stamp_file = f'{continuous_folder}/{ADC_name}/timestamps.npy'
ADC_continuous_time_stamp_data_raw = np.load(ADC_continuous_time_stamp_file, mmap_mode='r')

if is_convert_to_zero:
    ADC_continuous_time_stamp_data = (ADC_continuous_time_stamp_data_raw - ADC_continuous_time_stamp_data_raw[0])
else:
    ADC_continuous_time_stamp_data = ADC_continuous_time_stamp_data_raw

assert len(ADC_continuous_time_stamp_data) == len(
    photodiode_data), f"Length mismatch: {len(ADC_continuous_time_stamp_data)} vs {len(photodiode_data)}"

print(f"ADC_continuous_time_stamp_data: {ADC_continuous_time_stamp_data.shape}")
ADC_duration = len(photodiode_data) / ADC_sampling_rate
print(f"ADC duration: {ADC_duration} ")

edge_threshold = 1000
max_internal_gap = 300

raw_active = photodiode_data > edge_threshold

# ADC_DI_data_noGap = fill_short_gaps(
#     raw_active,
#     min_gap=max_internal_gap,
# )

ADC_DI_data_noGap = fill_short_gaps(~(fill_short_gaps(photodiode_data < 15000, min_gap=10)), min_gap=1000)

assert len(ADC_continuous_time_stamp_data) == len(
    ADC_DI_data_noGap), f"Length mismatch: {len(ADC_continuous_time_stamp_data)} vs {len(ADC_DI_data_noGap)}"

on_list_detected = detect_exposure_time(
    ADC_continuous_time_stamp_data,
    ADC_DI_data_noGap,
    threshold=detect_on_list_threshold,
    time_interval=False,
    is_inverted=is_signal_inverted,
    detection_target_label="on_list",
).ravel()

on_list_detected[-1] = (on_list_detected[-1] - 1) if on_list_detected[-1] == len(ADC_continuous_time_stamp_data) else \
    on_list_detected[-1]

on_list_time = ADC_continuous_time_stamp_data[on_list_detected]

(saved, stored_data), edited_target = sessionEdit.getSessionInfo(
    target=on_list_time
)

abnormal = []

for index, _ in enumerate(np.diff(on_list_time)):
    if _ < 0.08:
        abnormal.append(index)
    if _ > 0.11:
        abnormal.append(index)

start, end = gen_delete_by_range_index(abnormal, on_list_time)
# (start, end, frames_between)

deleteStartIndex = start
deleteEndIndex = end
# deleteEndIndex = None

if saved:
    on_list_time = edited_target
    print(f"Stored edits applied: {stored_data}")
else:
    if deleteFrames:
        on_list_time = np.delete(on_list_time, deleteFrames, )

    on_list_time = on_list_time[deleteStartIndex:deleteEndIndex]

    if interp_start:
        on_list_time = interp_replace(
            on_list_time,
            interp_start,
            interp_end,
            new_length=frames_between + 2,
        )

trials = loadmat(f"{base_dir}/{date}.mat")["trials"]

assert len(on_list_time) == len(trials), f"Length mismatch: {len(on_list_time)} vs {len(trials)}"

if not saved:
    print("Saving Data")
    if deleteFrames:
        success, message = sessionEdit.deleteByFrame(deleteFrames)
        if not success:
            print(message)

    if deleteStartIndex:
        success, message = sessionEdit.deleteByRange(deleteStartIndex, deleteEndIndex, )
        if not success:
            print(message)

    if interp_start:
        success, message = sessionEdit.interp(
            interp_start,
            interp_end,
            frames_between,
        )

        if not success:
            print(message)

print("Session edits stored")

on_list_time = np.append(on_list_time, on_list_time[-1] + 0.1)
# on_list_time = np.append(on_list_time[0] - 0.1, on_list_time)

np.save((npyfile_path := f"{base_dir}/data/on_list_times.npy"), on_list_time)
print(f"on_list_time saved to {npyfile_path}")

In [ ]:
print("first 10:", on_list_time[:10], "\n")

print("last 10:", on_list_time[-10:])

In [ ]:
for index, _ in enumerate(np.diff(on_list_time)):
    if _ < 0.08:
        print("Too short:")
        print(f"Gap detected at index {index}: {on_list_time[index]} to {on_list_time[index + 1]} (gap: {_})")
    if _ > 0.18:
        print("Too long:")
        print(f"Gap detected at index {index}: {on_list_time[index]} to {on_list_time[index + 1]} (gap: {_})")

In [ ]:
# start_time, end_time = (on_list_time[0] - 2), (on_list_time[2] + 1)  # Start
start_time, end_time = (on_list_time[-2] - 2), (on_list_time[-1] + 1)  # End

start_idx = np.searchsorted(ADC_continuous_time_stamp_data, start_time, side="left")
end_idx = np.searchsorted(ADC_continuous_time_stamp_data, end_time, side="right")

x = ADC_continuous_time_stamp_data[start_idx:end_idx]
y1 = photodiode_data[start_idx:end_idx]
y2 = ADC_DI_data_noGap[start_idx:end_idx]
edges_list = on_list_time

fig, ax1 = plt.subplots(figsize=(10, 4))

l1 = ax1.plot(x, y1, color='C0', lw=1, label='ADC')
ax1.set_ylim(0, 22000)
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("ADC")

ax2 = ax1.twinx()
l2 = ax2.plot(x, y2, color='C3', lw=2, label='Digital')
ax2.set_ylim(-0.1, 1.1)
ax2.set_ylabel("Digital")

# Add on_list rising/falling edge markers inside this displayed time window.
# on_list_time is already in the same time base as x.
left = np.searchsorted(edges_list, x[0], side="left")
right = np.searchsorted(edges_list, x[-1], side="right")
visible_edges = edges_list[left:right]

visible_edge_indices = np.searchsorted(ADC_continuous_time_stamp_data, visible_edges)

for t in visible_edges:
    ax1.axvline(t, color="orange", lw=0.8, alpha=0.6)
plt.show()

visible_edge_indices